# Cải thiện mô hình — v5

**Stage A1: DenseNet121 dưới pipeline robust**

Baseline v4 đã đóng băng và không bị notebook này chạm vào. Mọi manifest, fold,
định nghĩa preprocessing và chỉ số đều lấy nguyên từ đó, nên bảng so sánh mới
đặt cạnh bảng cũ được.

---

## Mốc cần vượt

`B1 = ResNet18 + stretch + augment mạnh + weighted CE`, mức filename-group trên
known benchmark:

| | |
|---|---:|
| AUC | 0,9779 |
| độ nhạy | 99,51% |
| **độ đặc hiệu** | **76,89%** |
| TN / FP / FN / TP | 173 / 52 / 1 / 202 |

Mục tiêu: **độ đặc hiệu 82–85%** trong khi giữ độ nhạy ≥97%. Trên 225 group
NORMAL, đó là giảm từ 52 xuống 33–40 ca báo nhầm.

## Thay đổi duy nhất về phương pháp: cách chọn checkpoint

v4 chọn epoch theo validation AUC. Chỉ số đó đã bão hòa ở 0,999x nên nó không
còn phân biệt được epoch nào tốt hơn — chọn theo nó là chọn theo nhiễu.

v5 chọn theo đúng thứ đang cần cải thiện:

```
chính:    độ đặc hiệu ở mức group, tại ngưỡng giữ độ nhạy ≥97%
hòa:      NLL không trọng số ở mức group thấp hơn
vẫn hòa:  giữ epoch sớm hơn
```

Chênh lệch độ đặc hiệu dưới 0,005 coi như hòa: ở mức group, một ca đổi phía đã
là 0,004, nên nhỏ hơn thế là nhiễu lấy mẫu.

Scheduler theo dõi NLL không trọng số. Không dùng loss có trọng số lớp để đánh
giá, vì trọng số làm lệch thang xác suất.

> **Review trước full run.** Bản này giữ nguyên câu hỏi Stage A1 nhưng bổ sung preflight cấu hình, cho scheduler đủ thời gian phát huy tác dụng, và lưu prediction OOF/benchmark theo tên mô hình để dùng cho hard-negative và ensemble về sau.

> **Về known benchmark.** Tập test này đã được xem nhiều lần trong các giai đoạn
> trước. Số liệu ở đây dùng để so sánh và cải tiến, **không** phải ước lượng
> khái quát hóa không thiên lệch.

## Cấu hình

In [1]:
RUN_MODE      = "auto"   # auto | smoke | full
DATA_ROOT_OVERRIDE = None # ví dụ: "/Users/me/data/chest_xray"
SEED          = 42
IMG_SIZE      = 224
BATCH_SIZE    = 32
EPOCHS        = 12
LR            = 1e-4
WEIGHT_DECAY  = 1e-5
PATIENCE      = 5         # phải lớn hơn scheduler patience để LR giảm còn có epoch phát huy
SCHEDULER_PATIENCE = 2
SCHEDULER_FACTOR   = 0.3
MIN_LR             = 1e-6
NUM_WORKERS   = 2         # runtime sẽ ép về 0 trên macOS

N_FOLDS       = 5         # 1 = một holdout 15%; 5 = cross-validation đầy đủ
VAL_FRACTION  = 0.15      # chỉ dùng khi N_FOLDS = 1
BORDER_FRAC   = 0.15      # dùng ở phần 4.3
DETERMINISTIC = True      # cudnn tất định; chậm hơn một chút, đổi lại tái lập tốt hơn
RESIZE_MODE   = "letterbox"  # mặc định cho thí nghiệm không ghi rõ "resize"
THRESHOLD_OBJECTIVE = "sensitivity"  # sensitivity | balanced_accuracy
CHECKPOINT_TIE_MARGIN = 0.005  # chênh độ đặc hiệu dưới mức này coi như bằng nhau
TARGET_SENSITIVITY = 0.97
BOOTSTRAP_REPS = 2000    # KTC cho audit tỉ lệ khung ở mức filename-derived group

# Hai bộ augmentation. "mạnh" mô phỏng thiết lập của các cài đặt công khai
# đạt độ đặc hiệu cao hơn: xoay 30 độ, zoom và dịch ảnh.
AUG_PRESETS = {
    "nhe":  {"rotation": 10, "scale": 0.00, "translate": 0.00, "jitter": 0.15},
    "manh": {"rotation": 30, "scale": 0.20, "translate": 0.10, "jitter": 0.20},
}

# Stage A1 chỉ chạy MỘT kiến trúc. Không thêm model khác vào cùng notebook để
# tránh thay đổi nhiều quyết định trước khi review kết quả DenseNet121.
# Một kiến trúc mỗi lần. So sánh chỉ công bằng khi mọi thứ khác giữ nguyên:
# cùng manifest, cùng fold, cùng preprocessing, cùng loss như B1 của v4.
ALL_EXPERIMENTS = [
    {"name": "densenet121_robust", "arch": "densenet121", "size": 224,
     "aug": "manh", "balancing": "weighted", "resize": "stretch",
     "hoi": "DenseNet121 dưới đúng pipeline đã giúp ResNet18"},
]

SMOKE_EXPERIMENTS = [
    {"name": "smoke_densenet", "arch": "densenet121", "size": 224,
     "aug": "manh", "balancing": "weighted", "resize": "stretch",
     "hoi": "kiểm tra pipeline, không dùng để báo cáo"},
]

# Mốc cần vượt: B1 của v4, mức filename-group trên known benchmark.
BASELINE_B1 = {"auc": 0.9779, "sensitivity": 0.9951, "specificity": 0.7689,
               "tn": 173, "fp": 52, "fn": 1, "tp": 202}

EXPERIMENTS = ALL_EXPERIMENTS
CLASSES = ("NORMAL", "PNEUMONIA")  # NORMAL=0, PNEUMONIA=1

EXPECTED_STAGE_A1 = {
    "arch": "densenet121", "size": 224, "resize": "stretch",
    "aug": "manh", "balancing": "weighted",
}

In [2]:
import gc, hashlib, json, os, platform, random, re, time, warnings
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import PIL
import scipy
import sklearn
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from PIL import Image
from sklearn.metrics import (average_precision_score, confusion_matrix, f1_score,
                             precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import StratifiedGroupKFold
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

warnings.filterwarnings("ignore", category=UserWarning)

IS_KAGGLE = Path("/kaggle/working").is_dir()
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

if RUN_MODE == "auto":
    RUN_MODE = "full" if IS_KAGGLE and DEVICE.type == "cuda" else "smoke"
if RUN_MODE not in {"smoke", "full"}:
    raise ValueError("RUN_MODE phải là 'auto', 'smoke' hoặc 'full'")

if RUN_MODE == "smoke":
    EPOCHS, N_FOLDS = 1, 1
    if DEVICE.type != "cuda":
        BATCH_SIZE = min(BATCH_SIZE, 16)
    EXPERIMENTS = SMOKE_EXPERIMENTS
else:
    EXPERIMENTS = ALL_EXPERIMENTS
    if len(EXPERIMENTS) != 1:
        raise AssertionError("Stage A1 full phải có đúng một thí nghiệm.")
    _actual = {k: EXPERIMENTS[0][k] for k in EXPECTED_STAGE_A1}
    if _actual != EXPECTED_STAGE_A1:
        raise AssertionError(
            f"Cấu hình Stage A1 bị lệch: {_actual}; kỳ vọng {EXPECTED_STAGE_A1}")

if platform.system() == "Darwin":
    NUM_WORKERS = 0  # notebook + spawn không an toàn với closure worker/cache global
LOCAL_PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                           if (p / ".git").is_dir()), Path.cwd())
WORK_DIR = (Path("/kaggle/working") if IS_KAGGLE
            else LOCAL_PROJECT_ROOT / "artifacts/notebook_rerun")
WORK_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = WORK_DIR / "train_log_v5.txt"
LOG_PATH.write_text("", encoding="utf-8")
PIN_MEMORY = DEVICE.type == "cuda"
AMP_ENABLED = DEVICE.type == "cuda"
DEVICE_NAME = (torch.cuda.get_device_name(0) if DEVICE.type == "cuda"
               else "Apple Metal (MPS)" if DEVICE.type == "mps" else platform.processor() or "CPU")

if DETERMINISTIC and DEVICE.type == "cuda":
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def set_seed(seed=SEED):
    """Seed mọi nguồn ngẫu nhiên mà pipeline đụng tới."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def loader_seed_args(seed=SEED):
    """generator + worker_init_fn cho DataLoader.

    Thiếu hai thứ này thì thứ tự xáo trộn và augmentation chạy trong worker vẫn
    ngẫu nhiên dù đã gọi set_seed — một lỗ hổng tái lập rất hay bị bỏ sót.
    """
    generator = torch.Generator()
    generator.manual_seed(seed)

    def worker_init_fn(worker_id):
        worker_seed = seed + worker_id
        random.seed(worker_seed)
        np.random.seed(worker_seed)

    return {"generator": generator, "worker_init_fn": worker_init_fn}


def log(*parts):
    """In ra màn hình, đồng thời ghi vào LOG_PATH.

    Output của notebook Kaggle có thể mất chunk khi in nhanh; file thì không.
    """
    line = " ".join(str(part) for part in parts)
    print(line)
    with open(LOG_PATH, "a", encoding="utf-8") as handle:
        handle.write(line + "\n")


set_seed()

# Ghi lại phiên bản thư viện. Thiếu nó thì con số trong báo cáo không gắn được
# với môi trường đã sinh ra chúng.
VERSIONS = {
    "python": platform.python_version(), "torch": torch.__version__,
    "torchvision": torchvision.__version__, "numpy": np.__version__,
    "pandas": pd.__version__, "scikit-learn": sklearn.__version__,
    "scipy": scipy.__version__,
    "pillow": PIL.__version__,
}
with open(WORK_DIR / "environment.json", "w") as handle:
    json.dump({**VERSIONS, "device": str(DEVICE), "device_name": DEVICE_NAME,
               "run_mode": RUN_MODE, "seed": SEED,
               "deterministic": DETERMINISTIC}, handle, indent=2)

# Lưu cấu hình đã resolve sau khi auto/smoke/full được áp dụng. File này giúp
# phân biệt source config với config thực sự sinh ra kết quả.
RESOLVED_CONFIG = {
    "run_mode": RUN_MODE,
    "seed": SEED,
    "image_cache_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "learning_rate": LR,
    "weight_decay": WEIGHT_DECAY,
    "patience": PATIENCE,
    "scheduler_patience": SCHEDULER_PATIENCE,
    "scheduler_factor": SCHEDULER_FACTOR,
    "min_lr": MIN_LR,
    "checkpoint_tie_margin": CHECKPOINT_TIE_MARGIN,
    "n_folds": N_FOLDS,
    "val_fraction": VAL_FRACTION,
    "deterministic": DETERMINISTIC,
    "threshold_objective": THRESHOLD_OBJECTIVE,
    "target_sensitivity": TARGET_SENSITIVITY,
    "bootstrap_reps": BOOTSTRAP_REPS,
    "augment_presets": AUG_PRESETS,
    "experiments": EXPERIMENTS,
}
with open(WORK_DIR / "resolved_config.json", "w", encoding="utf-8") as handle:
    json.dump(RESOLVED_CONFIG, handle, indent=2, ensure_ascii=False)

log("runtime:", "Kaggle" if IS_KAGGLE else "local", "| mode:", RUN_MODE)
log("device:", DEVICE, f"({DEVICE_NAME})", "| AMP:", AMP_ENABLED,
    "| workers:", NUM_WORKERS)
if RUN_MODE == "smoke":
    log("SMOKE RUN: chỉ kiểm tra pipeline; KHÔNG dùng chỉ số để báo cáo.")
log("phiên bản:", " ".join(f"{k}={v}" for k, v in VERSIONS.items()))

runtime: Kaggle | mode: full
device: cuda (Tesla T4) | AMP: True | workers: 2
phiên bản: python=3.12.13 torch=2.10.0+cu128 torchvision=0.25.0+cu128 numpy=2.0.2 pandas=2.3.3 scikit-learn=1.6.1 scipy=1.16.3 pillow=11.3.0


# 1. Dữ liệu

Giữ nguyên từ v4: cùng manifest, cùng group, cùng fold.

In [3]:
def list_images(directory):
    """Ảnh .jpeg thật, bỏ file sidecar ._* của macOS."""
    return sorted(p for p in Path(directory).glob("*.jpeg")
                  if not p.name.startswith("._"))


def find_data_root(search_paths):
    """Thư mục chứa trực tiếp train/NORMAL và train/PNEUMONIA."""
    if isinstance(search_paths, (str, Path)):
        search_paths = [search_paths]

    candidates = []
    for base in map(Path, search_paths):
        if not base.exists():
            continue
        for train_dir in base.rglob("train"):
            if "__MACOSX" in train_dir.parts:
                continue
            if (train_dir / "NORMAL").is_dir() and (train_dir / "PNEUMONIA").is_dir():
                candidates.append(train_dir.parent.resolve())

    if not candidates:
        raise FileNotFoundError(
            f"Không tìm thấy dataset dưới {[str(p) for p in search_paths]}. "
            "Kaggle: Add Data 'Chest X-Ray Images (Pneumonia)'. "
            "Mac: đặt DATA_ROOT_OVERRIDE hoặc CXR_DATA_ROOT.")

    candidates = sorted(set(candidates), key=lambda path: len(path.parts))
    for candidate in candidates:
        n = len(list_images(candidate / "train" / "NORMAL"))
        mark = "  <- dùng" if candidate == candidates[0] else "  (bản trùng, bỏ qua)"
        print(f"  {candidate}  [{n} ảnh train/NORMAL]{mark}")
    return candidates[0]


explicit_root = DATA_ROOT_OVERRIDE or os.environ.get("CXR_DATA_ROOT")
if explicit_root:
    DATA_ROOT = find_data_root([explicit_root])
else:
    cwd = Path.cwd()
    DATA_ROOT = find_data_root([
        "/kaggle/input", cwd / "chest_xray", cwd.parent / "chest_xray",
        cwd.parent.parent / "chest_xray", cwd / "data/raw",
        cwd.parent / "data/raw",
    ])
log("\nDATA_ROOT =", DATA_ROOT)

  /kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray  [1341 ảnh train/NORMAL]  <- dùng
  /kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray/chest_xray  [1341 ảnh train/NORMAL]  (bản trùng, bỏ qua)

DATA_ROOT = /kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray


In [4]:
PNEUMONIA_RE = re.compile(r"^person(\d+)_(bacteria|virus)_", re.IGNORECASE)
NORMAL_RE    = re.compile(r"^(?:(NORMAL\d+)-)?IM-(\d+)-", re.IGNORECASE)


def parse_group_id(filename):
    """Khoá group suy từ tên file; không khẳng định đây là clinical patient ID."""
    match = PNEUMONIA_RE.match(filename)
    if match:
        return f"pneumonia:{match.group(2).lower()}:{int(match.group(1))}"
    match = NORMAL_RE.match(filename)
    if match:
        return f"normal:{(match.group(1) or 'IM').lower()}:{int(match.group(2))}"
    raise ValueError(f"Tên file lạ, không suy ra được group: {filename}")


def build_manifest(root):
    """Một dòng cho mỗi ảnh: đường dẫn, split gốc, nhãn, filename-derived group."""
    rows = []
    for split in ("train", "val", "test"):
        for class_id, class_name in enumerate(CLASSES):
            directory = Path(root) / split / class_name
            if not directory.is_dir():
                continue
            for path in list_images(directory):
                rows.append({
                    "path": str(path.resolve()), "filename": path.name,
                    "split_original": split, "class_name": class_name,
                    "class_id": class_id, "group_id": parse_group_id(path.name)})
    if not rows:
        raise FileNotFoundError(f"Không có ảnh .jpeg nào dưới {root}")

    frame = pd.DataFrame(rows)
    frame["cache_index"] = np.arange(len(frame))   # vị trí trong cache ở mục 2.2
    return frame


manifest = build_manifest(DATA_ROOT)
log(f"{len(manifest):,} ảnh | {manifest['group_id'].nunique():,} filename-derived groups")
manifest.head(3)

5,856 ảnh | 4,097 filename-derived groups


,path,filename,split_original,class_name,class_id,group_id,cache_index
0,/kaggle/input/datasets/paultimothymooney/chest...,IM-0115-0001.jpeg,train,NORMAL,0,normal:im:115,0
1,/kaggle/input/datasets/paultimothymooney/chest...,IM-0117-0001.jpeg,train,NORMAL,0,normal:im:117,1
2,/kaggle/input/datasets/paultimothymooney/chest...,IM-0119-0001.jpeg,train,NORMAL,0,normal:im:119,2


## 1.2. Kiểm tra chất lượng dữ liệu

In [5]:
print("1.3.1  Số lượng theo split và lớp")
print("-" * 62)
print(f"{'split':<8}{'NORMAL':>9}{'PNEUMONIA':>12}{'tổng':>9}{'P/N':>7}")
for split in ("train", "val", "test"):
    subset = manifest[manifest["split_original"] == split]
    counts = subset["class_name"].value_counts()
    normal, pneumonia = int(counts.get("NORMAL", 0)), int(counts.get("PNEUMONIA", 0))
    ratio = pneumonia / normal if normal else float("nan")
    print(f"{split:<8}{normal:>9,}{pneumonia:>12,}{normal + pneumonia:>9,}{ratio:>7.2f}")
print(f"{'TỔNG':<8}{'':>9}{'':>12}{len(manifest):>9,}")

1.3.1  Số lượng theo split và lớp
--------------------------------------------------------------
split      NORMAL   PNEUMONIA     tổng    P/N
train       1,341       3,875    5,216   2.89
val             8           8       16   1.00
test          234         390      624   1.67
TỔNG                             5,856


In [6]:
print("1.3.2  Ảnh trùng nội dung (SHA-256)")
print("-" * 62)
manifest["sha256"] = [hashlib.sha256(Path(path).read_bytes()).hexdigest()
                      for path in manifest["path"]]
by_hash = defaultdict(list)
for digest, split in zip(manifest["sha256"], manifest["split_original"]):
    by_hash[digest].append(split)

duplicates = [s for s in by_hash.values() if len(s) > 1]
cross_split = [s for s in duplicates if len(set(s)) > 1]
print(f"tổng file             : {len(manifest):,}")
print(f"hash duy nhất         : {len(by_hash):,}")
print(f"nhóm ảnh trùng        : {len(duplicates)}")
print(f"  trong đó xuyên split: {len(cross_split)}")
print()
print("Không có ảnh y hệt nằm xuyên original split." if not cross_split
      else "CẢNH BÁO: ảnh trùng xuyên original split — kết quả đánh giá bị nhiễm.")

1.3.2  Ảnh trùng nội dung (SHA-256)
--------------------------------------------------------------
tổng file             : 5,856
hash duy nhất         : 5,824
nhóm ảnh trùng        : 30
  trong đó xuyên split: 0

Không có ảnh y hệt nằm xuyên original split.


In [7]:
print("1.3.4  Filename-derived group và nguy cơ trùng giữa các split")
print("-" * 62)
naive_re = re.compile(r"^(person\d+)_", re.IGNORECASE)
naive, corrected = defaultdict(set), defaultdict(set)
for filename, split in zip(manifest["filename"], manifest["split_original"]):
    match = naive_re.match(filename)
    naive[match.group(1).lower() if match else filename].add(split)
    corrected[parse_group_id(filename)].add(split)

naive_span     = sum(1 for s in naive.values() if len(s) > 1)
corrected_span = sum(1 for s in corrected.values() if len(s) > 1)
print(f"khoá person<N>              : {len(naive):,} nhóm, {naive_span} nằm ở >1 split")
print(f"khoá (phân nhóm, person<N>) : {len(corrected):,} nhóm, {corrected_span} nằm ở >1 split")

subtype_ids = defaultdict(set)
for filename in manifest["filename"]:
    match = PNEUMONIA_RE.match(filename)
    if match:
        subtype_ids[match.group(2).lower()].add(int(match.group(1)))
print()
for subtype, ids in sorted(subtype_ids.items()):
    print(f"  {subtype:<9}: {len(ids):,} số, dải 1..{max(ids)}, "
          f"mật độ {len(ids) / max(ids):.3f}")
print(f"  số được dùng bởi CẢ HAI phân nhóm: "
      f"{len(subtype_ids['bacteria'] & subtype_ids['virus']):,}")

per_group = Counter(parse_group_id(f) for f in manifest["filename"])
multi = sum(1 for n in per_group.values() if n > 1)
print(f"\nnhóm có >1 ảnh: {multi:,}/{len(per_group):,} "
      f"(nhiều nhất {max(per_group.values())} ảnh)")

1.3.4  Filename-derived group và nguy cơ trùng giữa các split
--------------------------------------------------------------
khoá person<N>              : 3,257 nhóm, 170 nằm ở >1 split
khoá (phân nhóm, person<N>) : 4,097 nhóm, 0 nằm ở >1 split

  bacteria : 1,437 số, dải 1..1954, mật độ 0.735
  virus    : 1,216 số, dải 1..1685, mật độ 0.722
  số được dùng bởi CẢ HAI phân nhóm: 979

nhóm có >1 ảnh: 726/4,097 (nhiều nhất 30 ảnh)


# 2. Phương pháp

In [8]:
def label_split(manifest, train, val, test):
    out = manifest.copy()
    out["split"] = pd.NA
    out.loc[train.index, "split"] = "train"
    out.loc[val.index,   "split"] = "val"
    out.loc[test.index,  "split"] = "test"
    return out


def make_folds(manifest, n_folds=N_FOLDS, val_fraction=VAL_FRACTION, seed=SEED):
    """Danh sách manifest, mỗi phần tử là một fold đã gán cột split."""
    pool = manifest[manifest["split_original"].isin(["train", "val"])]
    test = manifest[manifest["split_original"] == "test"]
    n_splits = max(2, round(1 / val_fraction)) if n_folds == 1 else n_folds
    splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)

    folds = [label_split(manifest, pool.iloc[train_idx], pool.iloc[val_idx], test)
             for train_idx, val_idx in
             splitter.split(pool, pool["class_id"], groups=pool["group_id"])]
    return folds[:1] if n_folds == 1 else folds


def count_leaked_groups(split):
    """Số filename-derived groups xuất hiện ở nhiều split. Phải bằng 0."""
    return int((split.groupby("group_id")["split"].nunique() > 1).sum())


def count_leaked_hashes(split):
    """Số nội dung ảnh y hệt xuất hiện ở nhiều split. Phải bằng 0."""
    return int((split.groupby("sha256")["split"].nunique() > 1).sum())


def split_summary(split):
    rows = []
    for name in ("train", "val", "test"):
        subset = split[split["split"] == name]
        counts = subset["class_name"].value_counts()
        normal, pneumonia = int(counts.get("NORMAL", 0)), int(counts.get("PNEUMONIA", 0))
        rows.append({"split": name, "NORMAL": normal, "PNEUMONIA": pneumonia,
                     "tổng": normal + pneumonia,
                     "groups": subset["group_id"].nunique(),
                     "P/N": round(pneumonia / max(normal, 1), 2)})
    return pd.DataFrame(rows).set_index("split")


FOLDS = make_folds(manifest)
log(f"\n{len(FOLDS)} fold, chia theo filename-derived group:")
for i, split in enumerate(FOLDS):
    s = split_summary(split)
    log(f"  fold {i}: train {s.loc['train','tổng']:>5,}  val {s.loc['val','tổng']:>4,}  "
        f"test {s.loc['test','tổng']:>4,}  |  group/hash ở >1 split: "
        f"{count_leaked_groups(split)}/{count_leaked_hashes(split)}")
    assert count_leaked_groups(split) == 0
    assert count_leaked_hashes(split) == 0
    split.to_csv(WORK_DIR / f"manifest_fold{i}.csv", index=False)

print("\nChi tiết fold 0:")
display(split_summary(FOLDS[0]))


5 fold, chia theo filename-derived group:
  fold 0: train 4,168  val 1,064  test  624  |  group/hash ở >1 split: 0/0
  fold 1: train 4,224  val 1,008  test  624  |  group/hash ở >1 split: 0/0
  fold 2: train 4,165  val 1,067  test  624  |  group/hash ở >1 split: 0/0
  fold 3: train 4,190  val 1,042  test  624  |  group/hash ở >1 split: 0/0
  fold 4: train 4,181  val 1,051  test  624  |  group/hash ở >1 split: 0/0

Chi tiết fold 0:


,NORMAL,PNEUMONIA,tổng,groups,P/N
split,,,,,
train,1092,3076,4168,2934,2.82
val,257,807,1064,735,3.14
test,234,390,624,428,1.67


## 2.2. Tiền xử lý và augmentation

In [9]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
MEAN_T = torch.tensor(IMAGENET_MEAN, device=DEVICE).view(1, 3, 1, 1)
STD_T  = torch.tensor(IMAGENET_STD,  device=DEVICE).view(1, 3, 1, 1)


def device_augment(batch, cfg):
    """Lật, xoay, zoom, dịch, đổi sáng/tương phản trên CUDA/MPS/CPU.

    Làm bằng PIL trong DataLoader thì CPU thành nút thắt: riêng RandomRotation
    và ColorJitter đã ngốn hơn 1.000 lần thời gian đọc cache. Ở đây mọi phép
    biến đổi là tensor op chạy theo lô trên accelerator đang chọn.

    Xoay, zoom và dịch được gộp vào MỘT phép biến đổi affine, nên chỉ nội suy
    một lần thay vì ba lần chồng lên nhau.
    """
    n, dev = batch.size(0), batch.device
    flip = torch.rand(n, device=dev) < 0.5
    batch = torch.where(flip.view(-1, 1, 1, 1), batch.flip(-1), batch)

    rand = lambda: torch.rand(n, device=dev) * 2 - 1          # noqa: E731  -1..1
    angles = rand() * (cfg["rotation"] * np.pi / 180)
    zoom = 1 + rand() * cfg["scale"]
    cos, sin = torch.cos(angles) * zoom, torch.sin(angles) * zoom
    theta = torch.zeros(n, 2, 3, device=dev)
    theta[:, 0, 0], theta[:, 0, 1] = cos, -sin
    theta[:, 1, 0], theta[:, 1, 1] = sin, cos
    theta[:, 0, 2] = rand() * cfg["translate"]
    theta[:, 1, 2] = rand() * cfg["translate"]
    grid = F.affine_grid(theta, batch.shape, align_corners=False)
    batch = F.grid_sample(batch, grid, align_corners=False, padding_mode="zeros")

    j = cfg["jitter"]
    scale = 1 + rand().view(-1, 1, 1, 1) * j
    contrast = 1 + rand().view(-1, 1, 1, 1) * j
    mean = batch.mean(dim=(1, 2, 3), keepdim=True)
    return ((batch * scale - mean) * contrast + mean).clamp(0, 1)


def to_model_input(batch_uint8, size=IMG_SIZE, aug=None):
    """(B,H,W) uint8 -> (B,3,H,W) chuẩn hoá trên DEVICE.

    Cache giữ ảnh ở IMG_SIZE; thí nghiệm nào cần kích thước khác thì thu nhỏ
    ngay trên DEVICE. Đây là resize hai bước (gốc -> IMG_SIZE -> size), áp dụng
    đồng nhất cho mọi split nên không tạo chênh lệch giữa train và test.
    """
    x = batch_uint8.to(DEVICE, non_blocking=True).float().div_(255).unsqueeze(1)
    if size != IMG_SIZE:
        x = F.interpolate(x, size=(size, size), mode="bilinear", align_corners=False)
    if aug is not None:
        x = device_augment(x, aug)
    return (x.expand(-1, 3, -1, -1) - MEAN_T) / STD_T


def resize_for_cache(image, size=IMG_SIZE, mode=RESIZE_MODE):
    gray = image.convert("L")
    if mode == "stretch":
        return np.asarray(gray.resize((size, size), Image.Resampling.BILINEAR))
    if mode != "letterbox":
        raise ValueError(f"RESIZE_MODE lạ: {mode}")
    gray.thumbnail((size, size), Image.Resampling.BILINEAR)
    array = np.asarray(gray)
    fill = int(np.median(array))
    canvas = Image.new("L", (size, size), color=fill)
    offset = ((size - gray.width) // 2, (size - gray.height) // 2)
    canvas.paste(gray, offset)
    return np.asarray(canvas)


def build_image_cache(manifest, size=IMG_SIZE, mode=RESIZE_MODE):
    cache = np.zeros((len(manifest), size, size), dtype=np.uint8)
    for position, path in enumerate(manifest["path"]):
        with Image.open(path) as image:
            cache[position] = resize_for_cache(image, size, mode)
        if (position + 1) % 1500 == 0:
            print(f"  {position + 1:,}/{len(manifest):,}")
    return cache


# Mỗi chế độ resize cần một cache riêng. Chỉ dựng những chế độ thực sự được
# dùng, để so sánh stretch với letterbox nằm trong cùng một lần chạy thay vì hai
# lần chạy khác nhau như trước.
REQUIRED_MODES = sorted({spec.get("resize", RESIZE_MODE) for spec in EXPERIMENTS})
IMAGE_CACHES = {}
for mode in REQUIRED_MODES:
    _started = time.time()
    IMAGE_CACHES[mode] = build_image_cache(manifest, mode=mode)
    log(f"cache {mode}: {IMAGE_CACHES[mode].nbytes / 1e6:.0f} MB cho "
        f"{len(manifest):,} ảnh trong {time.time() - _started:.0f}s")

# Cache mặc định cho các đoạn không gắn với một thí nghiệm cụ thể.
IMAGE_CACHE = IMAGE_CACHES[RESIZE_MODE if RESIZE_MODE in IMAGE_CACHES
                           else REQUIRED_MODES[0]]


class XRayDataset(Dataset):
    """Trả về ảnh uint8 thô; augmentation diễn ra trên DEVICE."""

    def __init__(self, rows, mode=RESIZE_MODE):
        self.rows, self.mode = rows, mode

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        cache_index, label = self.rows[index]
        return torch.from_numpy(IMAGE_CACHES[self.mode][cache_index]), label


def make_loader(split, name, batch_size=BATCH_SIZE, seed=SEED, mode=RESIZE_MODE):
    subset = split[split["split"] == name]
    return DataLoader(
        XRayDataset(list(zip(subset["cache_index"], subset["class_id"])), mode),
        batch_size=batch_size, shuffle=(name == "train"),
        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
        persistent_workers=NUM_WORKERS > 0,
        **loader_seed_args(seed))


def make_loaders(split, batch_size=BATCH_SIZE, seed=SEED, mode=RESIZE_MODE):
    return {name: make_loader(split, name, batch_size, seed, mode)
            for name in ("train", "val")}


def class_weights_from(split):
    """Trọng số nghịch tần suất, chuẩn hoá để loss giữ nguyên thang đo."""
    counts = Counter(split[split["split"] == "train"]["class_id"])
    total = sum(counts.values())
    return torch.tensor([total / (len(CLASSES) * counts[i]) for i in range(len(CLASSES))],
                        dtype=torch.float, device=DEVICE)

  1,500/5,856
  3,000/5,856
  4,500/5,856
cache stretch: 294 MB cho 5,856 ảnh trong 53s


## 2.3. Chỉ số đánh giá

In [10]:
METRIC_COLS = ["accuracy", "precision", "recall", "specificity",
               "f1", "bal_acc", "auc", "pr_auc"]


def metrics_at(labels, probs, threshold=0.5):
    """Chấm điểm tại một ngưỡng. threshold=0.5 chính là argmax trên hai logit."""
    labels, probs = np.asarray(labels), np.asarray(probs)
    preds = (probs >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(labels, preds, labels=[0, 1]).ravel()
    sensitivity, specificity = tp / max(tp + fn, 1), tn / max(tn + fp, 1)
    return {
        "threshold": float(threshold),
        "accuracy": float((labels == preds).mean()),
        "precision": precision_score(labels, preds, zero_division=0),
        "recall": recall_score(labels, preds, zero_division=0),
        "specificity": float(specificity),
        "f1": f1_score(labels, preds, zero_division=0),
        "bal_acc": float((sensitivity + specificity) / 2),
        "auc": roc_auc_score(labels, probs),
        "pr_auc": average_precision_score(labels, probs),
        "confusion_matrix": confusion_matrix(labels, preds).tolist(),
    }


def to_group_level(group_ids, labels, probs):
    """Gộp theo filename-derived group; xác suất là trung bình các ảnh."""
    frame = pd.DataFrame({"group": group_ids, "label": labels, "prob": probs})
    label_counts = frame.groupby("group")["label"].nunique()
    if int(label_counts.max()) != 1:
        bad = label_counts[label_counts > 1].index.tolist()[:5]
        raise ValueError(f"Group chứa nhiều nhãn, ví dụ: {bad}")
    rolled = frame.groupby("group", sort=True).agg(
        label=("label", "first"), prob=("prob", "mean"))
    return rolled["label"].to_numpy(), rolled["prob"].to_numpy()


def tune_threshold(labels, probs, objective=THRESHOLD_OBJECTIVE,
                   target_sensitivity=TARGET_SENSITIVITY):
    """Chọn một candidate thật trên validation/OOF, không nội suy qua vùng tie."""
    labels, probs = np.asarray(labels), np.asarray(probs)
    candidates = np.unique(np.clip(probs, 0.001, 0.999))
    if len(candidates) > 400:
        indices = np.linspace(0, len(candidates) - 1, 400).round().astype(int)
        candidates = candidates[np.unique(indices)]

    rows = [metrics_at(labels, probs, float(t)) for t in candidates]
    if objective == "balanced_accuracy":
        best = max(rows, key=lambda m: (m["bal_acc"], m["specificity"],
                                        m["recall"], m["threshold"]))
    elif objective == "sensitivity":
        feasible = [m for m in rows if m["recall"] >= target_sensitivity - 1e-12]
        if not feasible:
            warnings.warn("Không có ngưỡng đạt target sensitivity; dùng recall cao nhất.")
            feasible = rows
            best = max(feasible, key=lambda m: (m["recall"], m["specificity"],
                                                m["threshold"]))
        else:
            best = max(feasible, key=lambda m: (m["specificity"], m["bal_acc"],
                                                m["threshold"]))
    else:
        raise ValueError(f"THRESHOLD_OBJECTIVE lạ: {objective}")
    return float(best["threshold"]), best

## 2.4. Kiến trúc

In [11]:
class SmallCNN(nn.Module):
    """4 khối Conv-BN-ReLU-Pool rồi gộp toàn cục. Train từ đầu."""

    def __init__(self, num_classes=len(CLASSES)):
        super().__init__()
        def block(cin, cout):
            return nn.Sequential(
                nn.Conv2d(cin, cout, 3, padding=1), nn.BatchNorm2d(cout),
                nn.ReLU(inplace=True), nn.MaxPool2d(2))
        self.features = nn.Sequential(block(3, 32), block(32, 64),
                                      block(64, 128), block(128, 256))
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Dropout(0.3), nn.Linear(256, num_classes))

    def forward(self, x):
        return self.classifier(self.features(x))


def build_model(arch, pretrained=True, device=DEVICE):
    if arch == "resnet18":
        weights = models.ResNet18_Weights.DEFAULT if pretrained else None
        model = models.resnet18(weights=weights)
        model.fc = nn.Linear(model.fc.in_features, len(CLASSES))
    elif arch == "densenet121":
        weights = models.DenseNet121_Weights.DEFAULT if pretrained else None
        model = models.densenet121(weights=weights)
        model.classifier = nn.Linear(model.classifier.in_features, len(CLASSES))
    elif arch == "small_cnn":
        model = SmallCNN()
    else:
        raise ValueError(f"Kiến trúc lạ: {arch!r}")
    return model.to(device)


def target_layer_for(model, arch):
    """Lớp tích chập cuối để gắn Grad-CAM. Chỉ đích danh theo kiến trúc.

    Cách dò 'Conv2d cuối cùng' sẽ sai âm thầm khi đổi kiến trúc — heatmap vẫn
    hiện ra, chỉ là hiện sai chỗ.
    """
    return {"resnet18": lambda: model.layer4[-1],
            "densenet121": lambda: model.features.denseblock4,
            "small_cnn": lambda: model.features[-1]}[arch]()


def parameter_count_millions(arch):
    model = build_model(arch, pretrained=False, device=torch.device("cpu"))
    count = sum(p.numel() for p in model.parameters()) / 1e6
    del model
    return round(count, 1)


display(pd.DataFrame([
    {"thí nghiệm": s["name"], "kiến trúc": s["arch"], "px": s["size"],
     "resize": s.get("resize", RESIZE_MODE),
     "augment": s["aug"], "balancing": s["balancing"],
     "tham số (M)": parameter_count_millions(s["arch"]),
     "câu hỏi": s["hoi"]}
    for s in EXPERIMENTS]).set_index("thí nghiệm"))

,kiến trúc,px,resize,augment,balancing,tham số (M),câu hỏi
thí nghiệm,,,,,,,
densenet121_robust,densenet121,224,stretch,manh,weighted,7.0,DenseNet121 dưới đúng pipeline đã giúp ResNet18


## 2.5. Vòng huấn luyện

Quy tắc chọn checkpoint nằm trong `better_checkpoint`, khóa trước khi chạy.
Mỗi epoch ghi lại đủ AUC, PR-AUC, độ đặc hiệu tại ngưỡng, NLL, Brier và ngưỡng
đã chọn, xuất ra `epoch_history_*.csv` để truy ngược được vì sao một epoch được
giữ.

In [12]:
@torch.no_grad()
def predict(model, loader, size=IMG_SIZE):
    """(nhãn thật, xác suất PNEUMONIA) trên toàn bộ loader."""
    model.eval()
    labels_all, probs_all = [], []
    for images, labels in loader:
        logits = model(to_model_input(images, size))
        probs_all += torch.softmax(logits.float(), dim=1)[:, 1].cpu().tolist()
        labels_all += labels.tolist()
    return np.array(labels_all), np.array(probs_all)


def group_scores(labels, probs, groups):
    """Gộp dự đoán về mức filename-derived group."""
    frame = pd.DataFrame({"g": groups, "y": labels, "p": probs})
    label_counts = frame.groupby("g")["y"].nunique()
    if int(label_counts.max()) != 1:
        bad = label_counts[label_counts > 1].index.tolist()[:5]
        raise ValueError(f"Group validation chứa nhiều nhãn, ví dụ: {bad}")
    rolled = frame.groupby("g", sort=True).agg(y=("y", "first"), p=("p", "mean"))
    return rolled["y"].to_numpy(), rolled["p"].to_numpy()


def specificity_at_sensitivity(labels, probs, target=TARGET_SENSITIVITY):
    """Độ đặc hiệu cao nhất còn giữ được độ nhạy tối thiểu, kèm ngưỡng.

    Đây là chỉ số dự án đang thực sự cần cải thiện. Validation AUC đã bão hòa ở
    0,999x nên chọn epoch theo nó chỉ là chọn theo nhiễu.
    """
    positive = probs[labels == 1]
    if not len(positive):
        return 0.0, 0.5
    feasible = [c for c in np.unique(probs) if (positive >= c).mean() >= target]
    if not feasible:
        return 0.0, 0.0
    threshold = float(max(feasible))
    return float((probs[labels == 0] < threshold).mean()), threshold


def group_nll(labels, probs):
    """Log-loss không trọng số ở mức group.

    Không dùng loss có trọng số lớp để đánh giá: trọng số làm lệch thang xác
    suất, nên nó không nói được mô hình hiệu chuẩn tốt hay xấu.
    """
    p = np.clip(probs, 1e-7, 1 - 1e-7)
    return float(-np.mean(labels * np.log(p) + (1 - labels) * np.log(1 - p)))


def group_brier(labels, probs):
    """Brier score ở mức group."""
    return float(np.mean((probs - labels) ** 2))


def better_checkpoint(candidate, incumbent, margin=CHECKPOINT_TIE_MARGIN):
    """Quy tắc chọn checkpoint, khóa trước khi chạy.

    Độ đặc hiệu là chỉ số chính. Chênh lệch dưới ``margin`` coi như bằng nhau và
    NLL quyết định, vì độ đặc hiệu ở mức group nhảy bậc rời rạc — một ca đổi
    phía đã là 0,004 — nên chênh lệch nhỏ hơn thế là nhiễu lấy mẫu.

    Args:
        candidate: Chỉ số của epoch hiện tại.
        incumbent: Chỉ số của checkpoint đang giữ, hoặc None.
        margin: Ngưỡng coi hai độ đặc hiệu là bằng nhau.

    Returns:
        True nếu nên thay checkpoint.
    """
    if incumbent is None:
        return True
    gap = candidate["specificity"] - incumbent["specificity"]
    if gap > margin + 1e-12:
        return True
    if gap < -margin - 1e-12:
        return False
    # Hòa về độ đặc hiệu: lấy NLL thấp hơn. Vẫn hòa thì giữ epoch sớm hơn.
    return candidate["nll"] < incumbent["nll"] - 1e-9


def run_fold(spec, fold_index, epochs=EPOCHS):
    log(f"\n{'=' * 62}\n{spec['name']}  |  fold {fold_index}\n{'=' * 62}")
    resize = spec.get("resize", RESIZE_MODE)
    log(f"{spec['arch']} | {spec['size']}px | resize {resize} | "
        f"augment {spec['aug']} | balancing {spec['balancing']}")
    set_seed(SEED + fold_index)

    split = FOLDS[fold_index]
    loaders = make_loaders(split, seed=SEED + fold_index, mode=resize)
    try:
        model = build_model(spec["arch"], pretrained=True)
    except Exception as exc:
        raise RuntimeError("Không tải được ImageNet weights. Bật Internet trên Kaggle "
                           "hoặc tải weights vào cache trước khi chạy.") from exc
    size, aug = spec["size"], AUG_PRESETS[spec["aug"]]

    weights = class_weights_from(split) if spec["balancing"] == "weighted" else None
    log("trọng số lớp:", [round(w, 3) for w in weights.tolist()] if weights is not None
        else "không dùng")
    criterion = nn.CrossEntropyLoss(weight=weights)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    # Scheduler bám NLL không trọng số: nó phản ánh chất lượng xác suất, còn
    # AUC đã bão hòa và không còn phân biệt được epoch nào tốt hơn.
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=SCHEDULER_FACTOR,
        patience=SCHEDULER_PATIENCE, min_lr=MIN_LR)
    scaler = torch.amp.GradScaler("cuda", enabled=AMP_ENABLED)

    val_rows = split[split["split"] == "val"].reset_index(drop=True)
    val_groups = val_rows["group_id"].to_numpy()

    best, best_epoch, best_state, stale, history = None, 0, None, 0, []
    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        for images, labels in loaders["train"]:
            inputs = to_model_input(images, size, aug)
            labels = labels.to(DEVICE, non_blocking=PIN_MEMORY)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type=DEVICE.type, enabled=AMP_ENABLED):
                loss = criterion(model(inputs), labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            running_loss += loss.item() * inputs.size(0)

        train_loss = running_loss / len(loaders["train"].dataset)
        val_labels, val_probs = predict(model, loaders["val"], size)
        g_labels, g_probs = group_scores(val_labels, val_probs, val_groups)
        specificity, threshold = specificity_at_sensitivity(g_labels, g_probs)
        operating = metrics_at(g_labels, g_probs, threshold)
        (tn_v, fp_v), (fn_v, tp_v) = operating["confusion_matrix"]
        current = {
            "epoch": epoch, "train_loss": train_loss,
            "specificity": specificity, "sensitivity": operating["recall"],
            "threshold": threshold,
            "tn": int(tn_v), "fp": int(fp_v),
            "fn": int(fn_v), "tp": int(tp_v),
            "nll": group_nll(g_labels, g_probs),
            "brier": group_brier(g_labels, g_probs),
            "auc": roc_auc_score(g_labels, g_probs),
            "pr_auc": average_precision_score(g_labels, g_probs),
        }
        scheduler.step(current["nll"])
        current["lr"] = float(optimizer.param_groups[0]["lr"])
        history.append(current)

        marker = ""
        if better_checkpoint(current, best):
            best, best_epoch, stale = current, epoch, 0
            best_state = {k: v.detach().cpu().clone()
                          for k, v in model.state_dict().items()}
            marker = "  <- best"
        else:
            stale += 1

        log(f"epoch {epoch:>2}/{epochs}  loss {train_loss:.4f}  "
            f"spec@sens{TARGET_SENSITIVITY:.0%} {specificity:.4f}  "
            f"sens {current['sensitivity']:.4f}  "
            f"AUC {current['auc']:.4f}  PR {current['pr_auc']:.4f}  "
            f"NLL {current['nll']:.4f}  Brier {current['brier']:.4f}  "
            f"thr {threshold:.3f}  lr {current['lr']:.2e}{marker}")
        if stale >= PATIENCE:
            log(f"dừng sớm ở epoch {epoch}")
            break

    model.load_state_dict(best_state)
    log(f"khôi phục checkpoint epoch {best_epoch}, "
        f"spec {best['specificity']:.4f}, NLL {best['nll']:.4f}")
    val_labels, val_probs = predict(model, loaders["val"], size)
    assert np.array_equal(val_labels, val_rows["class_id"].to_numpy())

    tag = f"{spec['name']}_fold{fold_index}"
    checkpoint_path = WORK_DIR / f"{tag}.pth"
    torch.save(best_state, checkpoint_path)
    val_rows.assign(p_pneumonia=val_probs).to_csv(
        WORK_DIR / f"validation_predictions_{tag}.csv", index=False)
    pd.DataFrame(history).to_csv(
        WORK_DIR / f"epoch_history_{tag}.csv", index=False)

    result = {
        "experiment": spec["name"], "resize": resize, "arch": spec["arch"],
        "size": spec["size"], "balancing": spec["balancing"], "fold": fold_index,
        "best_epoch": best_epoch, "checkpoint": str(checkpoint_path),
        "val_labels": val_labels, "val_probs": val_probs, "val_groups": val_groups,
        "val": metrics_at(val_labels, val_probs, 0.5),
        "selection": best,
    }
    del model, loaders, optimizer, scheduler, scaler, best_state
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    elif DEVICE.type == "mps":
        torch.mps.empty_cache()
    return result

# 3. Kết quả

In [13]:
log(f"Bắt đầu {RUN_MODE}: {len(EXPERIMENTS)} thí nghiệm × {len(FOLDS)} fold × "
    f"tối đa {EPOCHS} epoch")
_t0 = time.time()
RUNS = [run_fold(spec, fold)
        for spec in EXPERIMENTS
        for fold in range(len(FOLDS))]
log(f"\ntổng thời gian: {(time.time() - _t0) / 60:.1f} phút "
    f"({len(EXPERIMENTS)} thí nghiệm × {len(FOLDS)} fold)")

Bắt đầu full: 1 thí nghiệm × 5 fold × tối đa 12 epoch

densenet121_robust  |  fold 0
densenet121 | 224px | resize stretch | augment manh | balancing weighted
Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 156MB/s]


trọng số lớp: [1.908, 0.678]
epoch  1/12  loss 0.1776  spec@sens97% 0.9588  sens 0.9715  AUC 0.9938  PR 0.9973  NLL 0.1587  Brier 0.0378  thr 0.134  lr 1.00e-04  <- best
epoch  2/12  loss 0.0969  spec@sens97% 0.9506  sens 0.9715  AUC 0.9936  PR 0.9972  NLL 0.2086  Brier 0.0493  thr 0.058  lr 1.00e-04
epoch  3/12  loss 0.0799  spec@sens97% 0.9877  sens 0.9715  AUC 0.9959  PR 0.9981  NLL 0.1411  Brier 0.0330  thr 0.231  lr 1.00e-04  <- best
epoch  4/12  loss 0.0657  spec@sens97% 0.9712  sens 0.9715  AUC 0.9938  PR 0.9972  NLL 0.1464  Brier 0.0335  thr 0.224  lr 1.00e-04
epoch  5/12  loss 0.0597  spec@sens97% 0.9918  sens 0.9715  AUC 0.9960  PR 0.9982  NLL 0.1469  Brier 0.0306  thr 0.214  lr 1.00e-04
epoch  6/12  loss 0.0538  spec@sens97% 0.9712  sens 0.9715  AUC 0.9968  PR 0.9985  NLL 0.1902  Brier 0.0469  thr 0.050  lr 3.00e-05
epoch  7/12  loss 0.0434  spec@sens97% 0.9835  sens 0.9715  AUC 0.9973  PR 0.9988  NLL 0.1260  Brier 0.0287  thr 0.205  lr 3.00e-05  <- best
epoch  8/12  loss 0.

In [14]:
display(pd.DataFrame([
    {"thí nghiệm": r["experiment"], "fold": r["fold"], "resize": r["resize"],
     "epoch tốt nhất": r["best_epoch"],
     "spec@sens97": round(r["selection"]["specificity"], 4),
     "group AUC": round(r["selection"]["auc"], 4),
     "NLL": round(r["selection"]["nll"], 4),
     "checkpoint": Path(r["checkpoint"]).name}
    for r in RUNS]).set_index(["thí nghiệm", "fold"]))

resize  epoch tốt nhất  spec@sens97  group AUC  \
thí nghiệm         fold                                                    
densenet121_robust 0     stretch              10       0.9918     0.9980   
                   1     stretch              12       0.9918     0.9991   
                   2     stretch               8       0.9959     0.9987   
                   3     stretch               7       0.9754     0.9967   
                   4     stretch               4       0.9959     0.9988   

                            NLL                    checkpoint  
thí nghiệm         fold                                        
densenet121_robust 0     0.0873  densenet121_robust_fold0.pth  
                   1     0.0361  densenet121_robust_fold1.pth  
                   2     0.0432  densenet121_robust_fold2.pth  
                   3     0.0810  densenet121_robust_fold3.pth  
                   4     0.0563  densenet121_robust_fold4.pth

## 3.2. Tổng hợp OOF

In [15]:
DEV_LABEL = "OOF validation" if len(FOLDS) > 1 else "validation holdout"
OOF, rows = {}, []
for spec in EXPERIMENTS:
    runs = sorted((r for r in RUNS if r["experiment"] == spec["name"]),
                  key=lambda r: r["fold"])
    labels = np.concatenate([r["val_labels"] for r in runs])
    probs = np.concatenate([r["val_probs"] for r in runs])
    groups = np.concatenate([r["val_groups"] for r in runs])
    folds_oof = np.concatenate([
        np.full(len(r["val_labels"]), r["fold"], dtype=int) for r in runs])
    group_labels, group_probs = to_group_level(groups, labels, probs)
    oof_images = pd.DataFrame({
        "group_id": groups, "label": labels, "p_pneumonia": probs,
        "fold": folds_oof, "experiment": spec["name"],
    })
    oof_images.to_csv(
        WORK_DIR / f"predictions_oof_{spec['name']}_images.csv", index=False)
    oof_groups = (oof_images.groupby("group_id", as_index=False)
                  .agg(label=("label", "first"),
                       p_pneumonia=("p_pneumonia", "mean"),
                       fold=("fold", "first")))
    if oof_groups["group_id"].duplicated().any():
        raise AssertionError("OOF group bị lặp sau khi gộp.")
    oof_groups["experiment"] = spec["name"]
    oof_groups.to_csv(
        WORK_DIR / f"predictions_oof_{spec['name']}_groups.csv", index=False)

    image_threshold, image_tuned = tune_threshold(labels, probs)
    group_threshold, group_tuned = tune_threshold(group_labels, group_probs)
    entry = {
        "labels": labels, "probs": probs, "groups": groups,
        "group_labels": group_labels, "group_probs": group_probs,
        "image_threshold": image_threshold, "group_threshold": group_threshold,
        "image_default": metrics_at(labels, probs, 0.5),
        "image_tuned": image_tuned,
        "group_default": metrics_at(group_labels, group_probs, 0.5),
        "group_tuned": group_tuned,
    }
    OOF[spec["name"]] = entry
    for unit, default_key, tuned_key in (
            ("image", "image_default", "image_tuned"),
            ("filename_group", "group_default", "group_tuned")):
        for mode, key in (("default_0.5", default_key), ("validation_tuned", tuned_key)):
            block = entry[key]
            rows.append({"experiment": spec["name"],
                         "resize": spec.get("resize", RESIZE_MODE),
                         "aug": spec["aug"], "balancing": spec["balancing"],
                         "arch": spec["arch"], "size": spec["size"],
                         "unit": unit, "mode": mode,
                         "threshold": block["threshold"],
                         **{k: block[k] for k in METRIC_COLS}})

oof_results = pd.DataFrame(rows)
oof_results.to_csv(WORK_DIR / "results_oof_validation.csv", index=False)
print(f"{DEV_LABEL.upper()} — không dùng test để chọn cấu hình\n")
display(oof_results.set_index(["experiment", "unit", "mode"]).round(4))

OOF VALIDATION — không dùng test để chọn cấu hình



resize   aug balancing  \
experiment         unit           mode                                        
densenet121_robust image          default_0.5       stretch  manh  weighted   
                                  validation_tuned  stretch  manh  weighted   
                   filename_group default_0.5       stretch  manh  weighted   
                                  validation_tuned  stretch  manh  weighted   

                                                           arch  size  \
experiment         unit           mode                                  
densenet121_robust image          default_0.5       densenet121   224   
                                  validation_tuned  densenet121   224   
                   filename_group default_0.5       densenet121   224   
                                  validation_tuned  densenet121   224   

                                                    threshold  accuracy  \
experiment         unit           mode                                    
densenet121_robust image          default_0.5          0.5000    0.9841   
                                  validation_tuned     0.7020    0.9776   
                   filename_group default_0.5          0.5000    0.9804   
                                  validation_tuned     0.6389    0.9771   

                                                    precision  recall  \
experiment         unit           mode                                  
densenet121_robust image          default_0.5          0.9956  0.9830   
                                  validation_tuned     0.9976  0.9722   
                   filename_group default_0.5          0.9930  0.9776   
                                  validation_tuned     0.9954  0.9702   

                                                    specificity      f1  \
experiment         unit           mode                                    
densenet121_robust image          default_0.5            0.9874  0.9892   
                                  validation_tuned       0.9933  0.9847   
                   filename_group default_0.5            0.9861  0.9852   
                                  validation_tuned       0.9910  0.9826   

                                                    bal_acc     auc  pr_auc  
experiment         unit           mode                                       
densenet121_robust image          default_0.5        0.9852  0.9985  0.9995  
                                  validation_tuned   0.9828  0.9985  0.9995  
                   filename_group default_0.5        0.9818  0.9979  0.9990  
                                  validation_tuned   0.9806  0.9979  0.9990

## 3.3. Khóa cấu hình

In [16]:
if len(OOF) != 1:
    raise AssertionError("Stage A1 phải có đúng một cấu hình OOF.")
BEST = next(iter(OOF))
BEST_SPEC = next(spec for spec in EXPERIMENTS if spec["name"] == BEST)
IMAGE_THRESHOLD = OOF[BEST]["image_threshold"]
GROUP_THRESHOLD = OOF[BEST]["group_threshold"]

print(f"Cấu hình Stage A1 duy nhất; ngưỡng đã khóa bằng {DEV_LABEL}: {BEST}")
print(f"  group AUC          : {OOF[BEST]['group_default']['auc']:.4f}")
print(f"  threshold mức ảnh  : {IMAGE_THRESHOLD:.4f}")
print(f"  threshold mức group: {GROUP_THRESHOLD:.4f}")
print(f"  objective          : {THRESHOLD_OBJECTIVE}",
      f"(sensitivity >= {TARGET_SENSITIVITY:.0%})"
      if THRESHOLD_OBJECTIVE == "sensitivity" else "")
assert "test_probs" not in RUNS[0], "Test đã bị đọc trước khi khóa cấu hình"
# Một nguồn sự thật cho khâu tiền xử lý của cấu hình đã khóa. Mọi thứ phía sau
# (test, Grad-CAM, che vùng, mục 4.5) phải dùng đúng biến này.
BEST_RESIZE = BEST_SPEC.get("resize", RESIZE_MODE)
XAI_RESIZE = BEST_RESIZE
XAI_CACHE = IMAGE_CACHES[BEST_RESIZE]
print(f"  tiền xử lý         : {BEST_RESIZE}")

Cấu hình Stage A1 duy nhất; ngưỡng đã khóa bằng OOF validation: densenet121_robust
  group AUC          : 0.9979
  threshold mức ảnh  : 0.7020
  threshold mức group: 0.6389
  objective          : sensitivity (sensitivity >= 97%)
  tiền xử lý         : stretch


## 3.4. Known benchmark

Chỉ chạy sau khi cấu hình đã khóa bằng OOF.

In [17]:
def show_confusion(title, matrix):
    (tn, fp), (fn, tp) = matrix
    sensitivity, specificity = tp / max(tp + fn, 1), tn / max(tn + fp, 1)
    precision = tp / max(tp + fp, 1)
    print(f"\n{title}  (n={tn + fp + fn + tp})")
    print(f"  TN {tn:>4}   FP {fp:>4}")
    print(f"  FN {fn:>4}   TP {tp:>4}")
    print(f"  bỏ sót {fn}/{fn + tp} ca viêm phổi  → độ nhạy {sensitivity:.1%}")
    print(f"  báo nhầm {fp}/{tn + fp} ca bình thường → độ đặc hiệu {specificity:.1%}")
    print(f"  precision {precision:.1%}  |  balanced accuracy "
          f"{(sensitivity + specificity) / 2:.1%}")


def load_model_from_run(run):
    model = build_model(run["arch"], pretrained=False)
    try:
        state = torch.load(run["checkpoint"], map_location="cpu", weights_only=True)
    except TypeError:  # PyTorch cũ
        state = torch.load(run["checkpoint"], map_location="cpu")
    model.load_state_dict(state)
    return model.eval()


best_runs = sorted((r for r in RUNS if r["experiment"] == BEST),
                   key=lambda r: r["fold"])
test_rows = FOLDS[0][FOLDS[0]["split"] == "test"].reset_index(drop=True)

# Ảnh test phải qua đúng khâu tiền xử lý mà mô hình đã được train. Dùng mặc
# định letterbox ở đây sẽ chấm một mô hình stretch trên phân phối đầu vào khác
# hẳn lúc train, và mọi chỉ số phía dưới đều sai mà không có dấu hiệu gì.
assert {run["resize"] for run in best_runs} == {BEST_RESIZE}, \
    "Các fold của BEST không cùng một chế độ resize."
test_loader = make_loader(FOLDS[0], "test", seed=SEED, mode=BEST_RESIZE)
print(f"Tiền xử lý dùng cho known benchmark test: {BEST_RESIZE}\n")

test_probabilities, test_labels = [], None
for run in best_runs:
    model = load_model_from_run(run)
    labels, probs = predict(model, test_loader, run["size"])
    if test_labels is None:
        test_labels = labels
    else:
        assert np.array_equal(test_labels, labels)
    test_probabilities.append(probs)
    del model
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    elif DEVICE.type == "mps":
        torch.mps.empty_cache()

test_probs = np.mean(test_probabilities, axis=0)
assert np.array_equal(test_labels, test_rows["class_id"].to_numpy())
test_groups = test_rows["group_id"].to_numpy()
group_labels, group_probs = to_group_level(test_groups, test_labels, test_probs)

FINAL = {
    "image_default": metrics_at(test_labels, test_probs, 0.5),
    "image_tuned": metrics_at(test_labels, test_probs, IMAGE_THRESHOLD),
    "group_default": metrics_at(group_labels, group_probs, 0.5),
    "group_tuned": metrics_at(group_labels, group_probs, GROUP_THRESHOLD),
    "test_labels": test_labels, "test_probs": test_probs,
    "group_labels": group_labels, "group_probs": group_probs,
}

final_rows = []
for unit, default_key, tuned_key in (
        ("image", "image_default", "image_tuned"),
        ("filename_group", "group_default", "group_tuned")):
    for mode, key in (("default_0.5", default_key), ("validation_tuned", tuned_key)):
        block = FINAL[key]
        final_rows.append({"experiment": BEST,
                           "arch": BEST_SPEC["arch"],
                           "resize": BEST_RESIZE,
                           "aug": BEST_SPEC["aug"],
                           "balancing": BEST_SPEC["balancing"],
                           "evaluation": "known_benchmark_not_final",
                           "unit": unit, "mode": mode,
                           "threshold": block["threshold"],
                           **{k: block[k] for k in METRIC_COLS},
                           "confusion_matrix": json.dumps(block["confusion_matrix"])})
final_results = pd.DataFrame(final_rows)
final_results.to_csv(WORK_DIR / "results_known_benchmark_test.csv", index=False)
benchmark_images = test_rows.assign(
    p_pneumonia=test_probs,
    pred=(test_probs >= IMAGE_THRESHOLD).astype(int),
    experiment=BEST, resize=BEST_RESIZE)
benchmark_images.to_csv(
    WORK_DIR / f"predictions_known_benchmark_{BEST}_images.csv", index=False)
benchmark_images.to_csv(
    WORK_DIR / "predictions_known_benchmark_test_images.csv", index=False)

best_group_predictions = (pd.DataFrame({
    "group_id": test_groups, "label": test_labels, "p_pneumonia": test_probs,
}).groupby("group_id", as_index=False)
  .agg(label=("label", "first"), p_pneumonia=("p_pneumonia", "mean")))
best_group_predictions["pred"] = (
    best_group_predictions["p_pneumonia"] >= GROUP_THRESHOLD).astype(int)
best_group_predictions["experiment"] = BEST
best_group_predictions["resize"] = BEST_RESIZE
best_group_predictions.to_csv(
    WORK_DIR / f"predictions_known_benchmark_{BEST}_groups.csv", index=False)
best_group_predictions.to_csv(
    WORK_DIR / "predictions_known_benchmark_test_groups.csv", index=False)

print(f"KNOWN BENCHMARK TEST — cấu hình đã khóa: {BEST}\n")
display(final_results.set_index(["unit", "mode"]).round(4))
show_confusion(f"{BEST} — TEST image @ {IMAGE_THRESHOLD:.3f}",
               FINAL["image_tuned"]["confusion_matrix"])
show_confusion(f"{BEST} — TEST filename-group @ {GROUP_THRESHOLD:.3f}",
               FINAL["group_tuned"]["confusion_matrix"])

# Loader BEST không còn dùng; giải phóng persistent workers trước mục 3.5.
del test_loader
gc.collect()


Tiền xử lý dùng cho known benchmark test: stretch

KNOWN BENCHMARK TEST — cấu hình đã khóa: densenet121_robust



experiment         arch   resize  \
unit           mode                                                         
image          default_0.5       densenet121_robust  densenet121  stretch   
               validation_tuned  densenet121_robust  densenet121  stretch   
filename_group default_0.5       densenet121_robust  densenet121  stretch   
               validation_tuned  densenet121_robust  densenet121  stretch   

                                  aug balancing                 evaluation  \
unit           mode                                                          
image          default_0.5       manh  weighted  known_benchmark_not_final   
               validation_tuned  manh  weighted  known_benchmark_not_final   
filename_group default_0.5       manh  weighted  known_benchmark_not_final   
               validation_tuned  manh  weighted  known_benchmark_not_final   

                                 threshold  accuracy  precision  recall  \
unit           mode                                                       
image          default_0.5          0.5000    0.9087     0.8742  0.9974   
               validation_tuned     0.7020    0.9343     0.9087  0.9949   
filename_group default_0.5          0.5000    0.8738     0.7922  0.9951   
               validation_tuned     0.6389    0.8949     0.8211  0.9951   

                                 specificity      f1  bal_acc     auc  pr_auc  \
unit           mode                                                             
image          default_0.5            0.7607  0.9317   0.8791  0.9846  0.9884   
               validation_tuned       0.8333  0.9498   0.9141  0.9846  0.9884   
filename_group default_0.5            0.7644  0.8821   0.8798  0.9801  0.9705   
               validation_tuned       0.8044  0.8998   0.8998  0.9801  0.9705   

                                      confusion_matrix  
unit           mode                                     
image          default_0.5       [[178, 56], [1, 389]]  
               validation_tuned  [[195, 39], [2, 388]]  
filename_group default_0.5       [[172, 53], [1, 202]]  
               validation_tuned  [[181, 44], [1, 202]]


densenet121_robust — TEST image @ 0.702  (n=624)
  TN  195   FP   39
  FN    2   TP  388
  bỏ sót 2/390 ca viêm phổi  → độ nhạy 99.5%
  báo nhầm 39/234 ca bình thường → độ đặc hiệu 83.3%
  precision 90.9%  |  balanced accuracy 91.4%

densenet121_robust — TEST filename-group @ 0.639  (n=428)
  TN  181   FP   44
  FN    1   TP  202
  bỏ sót 1/203 ca viêm phổi  → độ nhạy 99.5%
  báo nhầm 44/225 ca bình thường → độ đặc hiệu 80.4%
  precision 82.1%  |  balanced accuracy 90.0%


0

## 3.5. So với B1

Bảng dưới đặt DenseNet121 cạnh mốc B1 của v4. Điều kiện giữ DenseNet121, thống
nhất trước khi chạy:

- độ đặc hiệu tăng ít nhất **0,02** mà độ nhạy không xuống dưới 0,97; **hoặc**
- AUC tăng và độ đặc hiệu không thấp hơn B1 quá 0,01.

In [18]:
group_row = FINAL["group_tuned"]
predictions = (FINAL["group_probs"] >= GROUP_THRESHOLD).astype(int)
tn, fp, fn, tp = confusion_matrix(FINAL["group_labels"], predictions,
                                  labels=[0, 1]).ravel()

comparison = pd.DataFrame([
    {"model": "B1 resnet18 (v4)", **BASELINE_B1},
    {"model": f"{BEST} (v5)", "auc": group_row["auc"],
     "sensitivity": group_row["recall"], "specificity": group_row["specificity"],
     "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)},
])
display(comparison.round(4))

delta_specificity = group_row["specificity"] - BASELINE_B1["specificity"]
print(f"\nchênh độ đặc hiệu so với B1 : {delta_specificity:+.4f}")
print(f"chênh AUC so với B1        : {group_row['auc'] - BASELINE_B1['auc']:+.4f}")
print(f"ca báo nhầm tránh được     : {BASELINE_B1['fp'] - int(fp):+d}")
print(f"độ nhạy                    : {group_row['recall']:.4f}"
      f"  ({'đạt' if group_row['recall'] >= 0.97 else 'KHÔNG ĐẠT'} ngưỡng 0.97)")

keep = ((delta_specificity >= 0.02 and group_row["recall"] >= 0.97)
        or (group_row["auc"] > BASELINE_B1["auc"] and delta_specificity >= -0.01))
decision = {
    "evaluation_role": "engineering_decision_on_known_benchmark",
    "run_mode": RUN_MODE, "model": BEST, "keep": bool(keep),
    "delta_specificity_vs_b1": float(delta_specificity),
    "delta_auc_vs_b1": float(group_row["auc"] - BASELINE_B1["auc"]),
    "fp_avoided_vs_b1": int(BASELINE_B1["fp"] - int(fp)),
    "sensitivity": float(group_row["recall"]),
}
with open(WORK_DIR / "stage_a1_decision.json", "w", encoding="utf-8") as handle:
    json.dump(decision, handle, indent=2, ensure_ascii=False)
print(f"\n=> {'GIỮ' if keep else 'KHÔNG GIỮ'} DenseNet121 "
      "theo tiêu chí engineering đã đặt trước.")
if RUN_MODE == "smoke":
    print("   (smoke run — con số chỉ để kiểm tra pipeline, không dùng để quyết định)")

,model,auc,sensitivity,specificity,tn,fp,fn,tp
0,B1 resnet18 (v4),0.9779,0.9951,0.7689,173,52,1,202
1,densenet121_robust (v5),0.9801,0.9951,0.8044,181,44,1,202



chênh độ đặc hiệu so với B1 : +0.0355
chênh AUC so với B1        : +0.0022
ca báo nhầm tránh được     : +8
độ nhạy                    : 0.9951  (đạt ngưỡng 0.97)

=> GIỮ DenseNet121 theo tiêu chí engineering đã đặt trước.


# 4. Bước tiếp theo

Chưa triển khai trong notebook này, theo đúng thứ tự đã thống nhất:

1. DeiT-Small dưới cùng pipeline;
2. hard-negative fine-tuning trên mô hình đơn tốt nhất;
3. ensemble từ prediction đã lưu.

Mỗi bước là một lần chạy riêng, review xong mới sang bước sau.